# 01 DataCheck

## 0. Research Questions

**Research question:** Is the proportion of current cigarette use different between students who
felt sad or hopeless and those who did not?

-----------------------------------------------------------------------------------------------

**Additional EDA:** Do Sad/Hopeless students show stronger clustering of substance-related health-risk behaviors?

Note: In this extension, health-risk behavior clustering refers to students reporting multiple health-risk behaviors at the same time, including current cigarette use, current alcohol use, and current marijuana use.

## 1. Data

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
ROOT = Path.cwd().resolve().parent
RAW_PATH = ROOT / "data" / "raw" / "YRBS_2007.csv"

PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

Q8_PROCESSED_PATH = PROCESSED_DIR / "yrbs_cycle3_q8_processed_only.csv"
HEALTH_RISK_PROCESSED_PATH = PROCESSED_DIR / "yrbs_cycle3_health_risk_processed_only.csv"

TAB_DIR = ROOT / "outputs" / "tables"
TAB_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)

# ------------------------------------------------------------
# Load raw data
# ------------------------------------------------------------
raw = pd.read_csv(RAW_PATH)

# recoded only stores processed variables
recoded = pd.DataFrame(index=raw.index)

print("Rows, columns:", raw.shape)

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------
def recode_binary(series, yes_codes, no_codes):
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    result.loc[series.isin(no_codes)] = 0
    result.loc[series.isin(yes_codes)] = 1
    return result


def raw_code_count_table(series, label_map, output_name):
    valid_codes = list(label_map.keys())

    table = pd.DataFrame({
        "code": valid_codes + ["Missing"],
        "count": [int(series.eq(code).sum()) for code in valid_codes]
                 + [int((series.isna() | ~series.isin(valid_codes)).sum())],
        "meaning": list(label_map.values()) + ["Missing or invalid"]
    })

    table.to_csv(TAB_DIR / output_name, index=False)
    display(table)
    return table


def save_processed_only(data, cols, required_cols, path, label):
    missing_cols = [col for col in cols if col not in data.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns for {label}: {missing_cols}")

    processed = data.dropna(subset=required_cols).copy()
    processed = processed[cols].copy()

    for col in processed.columns:
        if col.endswith("_binary") or col == "health_risk_behavior_count":
            processed[col] = processed[col].astype(int)

    processed.to_csv(path, index=False)

    print(f"Rows saved for {label}:", len(processed))
    print("Saved to:", path)

    return processed

Rows, columns: (14041, 103)


## 2. Variable Definitions

### 2.1 Group variable: SadOrHopeless

- **Variable name:** `SadOrHopeless`
- **What the variable measures:** whether the student felt so sad or hopeless almost every day for 2 weeks or more in a row during the past 12 months that they stopped doing some usual activities.
- **Valid codes used:**  
  - `1` = Yes  
  - `2` = No
- **Recoding rule used in this project:**  
  - `1` -> `sad_binary = 1`  
  - `2` -> `sad_binary = 0`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.
- **Final valid sample size for proportion  analysis:** 13845

In [8]:
sad_raw = raw["SadOrHopeless"]

# ------------------------------------------------------------
# Recode SadOrHopeless
# 1 = felt sad or hopeless
# 2 = did not feel sad or hopeless
# ------------------------------------------------------------
recoded["sad_binary"] = recode_binary(
    sad_raw,
    yes_codes=[1],
    no_codes=[2]
)

recoded["sad_group"] = recoded["sad_binary"].map({
    1: "Sad/Hopeless: Yes",
    0: "Sad/Hopeless: No"
})

# ------------------------------------------------------------
# Data check table
# ------------------------------------------------------------
valid_sad_codes = [1, 2]

sad_check = pd.DataFrame({
    "metric": [
        "Total rows",
        "Missing values",
        "Non-missing values",
        "Code 1 count",
        "Code 2 count",
        "Invalid non-missing values",
        "Final valid sample size"
    ],
    "value": [
        len(raw),
        int(sad_raw.isna().sum()),
        int(sad_raw.notna().sum()),
        int(sad_raw.eq(1).sum()),
        int(sad_raw.eq(2).sum()),
        int((sad_raw.notna() & ~sad_raw.isin(valid_sad_codes)).sum()),
        int(sad_raw.isin(valid_sad_codes).sum())
    ]
})

sad_check.to_csv(TAB_DIR / "01_sad_or_hopeless_data_check.csv", index=False)
display(sad_check)

,metric,value
0,Total rows,14041
1,Missing values,196
2,Non-missing values,13845
3,Code 1 count,4153
4,Code 2 count,9692
5,Invalid non-missing values,0
6,Final valid sample size,13845


### 2.2 Response variable: CurrentCigaretteUse
- **Variable name:** `CurrentCigaretteUse`
- **What the variable measures:** number of days the student smoked cigarettes during the past 30 days.
- **Valid codes used:**  
  - `1` = 0 days  
  - `2` = 1 or 2 days  
  - `3` = 3 to 5 days  
  - `4` = 6 to 9 days  
  - `5` = 10 to 19 days  
  - `6` = 20 to 29 days  
  - `7` = all 30 days
- **Recoding rule used in this project:**  
  - `1` -> `smoker_binary = 0` (non-smoker in the past 30 days)  
  - `2` to `7` -> `smoker_binary = 1` (reported smoking on at least 1 day in the past 30 days)
- **Grouped version for EDA**
  - `1` -> `Non-smoker (0 days)`
  - `2, 3` -> `Light (1~5 days)`  
  - `4, 5` -> `Moderate (6~19 days)`  
  - `6, 7` -> `Frequent (20~30 days)`
- **How missing or invalid values are handled:** missing or invalid values are excluded from the analysis.
- **Final valid sample size for proportion analysis:** 13323

In [9]:
current_cig_raw = raw["CurrentCigaretteUse"]

# ------------------------------------------------------------
# Recode CurrentCigaretteUse
# 1 = 0 days -> non-smoker
# 2-7 = smoked at least 1 day -> current smoker
# ------------------------------------------------------------
recoded["smoker_binary"] = recode_binary(
    current_cig_raw,
    yes_codes=[2, 3, 4, 5, 6, 7],
    no_codes=[1]
)

recoded["current_cigarette_status"] = recoded["smoker_binary"].map({
    0: "Non-smoker",
    1: "Current smoker"
})

# ------------------------------------------------------------
# Smoking frequency group for EDA
# ------------------------------------------------------------
recoded["smoking_freq_group"] = pd.NA

recoded.loc[current_cig_raw.eq(1), "smoking_freq_group"] = "Non-smoker (0 days)"
recoded.loc[current_cig_raw.isin([2, 3]), "smoking_freq_group"] = "Light (1~5 days)"
recoded.loc[current_cig_raw.isin([4, 5]), "smoking_freq_group"] = "Moderate (6~19 days)"
recoded.loc[current_cig_raw.isin([6, 7]), "smoking_freq_group"] = "Frequent (20~30 days)"

# ------------------------------------------------------------
# Raw code count table
# ------------------------------------------------------------
current_cig_label_map = {
    1: "0 days",
    2: "1 or 2 days",
    3: "3 to 5 days",
    4: "6 to 9 days",
    5: "10 to 19 days",
    6: "20 to 29 days",
    7: "all 30 days"
}

raw_code_count_table(
    current_cig_raw,
    current_cig_label_map,
    "01_current_cigarette_use_code_counts.csv"
)

# ------------------------------------------------------------
# Smoking frequency group count table
# ------------------------------------------------------------
freq_order = [
    "Non-smoker (0 days)",
    "Light (1~5 days)",
    "Moderate (6~19 days)",
    "Frequent (20~30 days)",
    "Missing"
]

smoking_freq_group_counts = (
    recoded["smoking_freq_group"]
    .fillna("Missing")
    .value_counts()
    .reindex(freq_order, fill_value=0)
    .rename_axis("smoking_freq_group")
    .reset_index(name="count")
)

smoking_freq_group_counts.to_csv(TAB_DIR / "01_smoking_freq_group_counts.csv", index=False)
display(smoking_freq_group_counts)

,code,count,meaning
0,1,10734,0 days
1,2,753,1 or 2 days
2,3,375,3 to 5 days
3,4,250,6 to 9 days
4,5,295,10 to 19 days
5,6,229,20 to 29 days
6,7,687,all 30 days
7,Missing,718,Missing or invalid


,smoking_freq_group,count
0,Non-smoker (0 days),10734
1,Light (1~5 days),1128
2,Moderate (6~19 days),545
3,Frequent (20~30 days),916
4,Missing,718


## 3. Additional EDA: Variable Definition and Coding

### 3.1 Additional Variables: Alcohol and Marijuana

In [10]:
# ------------------------------------------------------------
# Additional variables: CurrentAlcoholUse and CurrentMarijuaUse
# ------------------------------------------------------------

alcohol_raw = raw["CurrentAlcoholUse"]
marijuana_raw = raw["CurrentMarijuaUse"]

# ------------------------------------------------------------
# Recode CurrentAlcoholUse
# 1 = 0 days
# 2-7 = current alcohol use
# ------------------------------------------------------------
recoded["alcohol_binary"] = recode_binary(
    alcohol_raw,
    yes_codes=[2, 3, 4, 5, 6, 7],
    no_codes=[1]
)

# ------------------------------------------------------------
# Recode CurrentMarijuaUse
# 1 = 0 times
# 2-6 = current marijuana use
# ------------------------------------------------------------
recoded["marijuana_binary"] = recode_binary(
    marijuana_raw,
    yes_codes=[2, 3, 4, 5, 6],
    no_codes=[1]
)

# ------------------------------------------------------------
# Combined raw code count table
# ------------------------------------------------------------
alcohol_label_map = {
    1: "0 days",
    2: "1 or 2 days",
    3: "3 to 5 days",
    4: "6 to 9 days",
    5: "10 to 19 days",
    6: "20 to 29 days",
    7: "all 30 days"
}

marijuana_label_map = {
    1: "0 times",
    2: "1 or 2 times",
    3: "3 to 9 times",
    4: "10 to 19 times",
    5: "20 to 39 times",
    6: "40 or more times"
}

def make_code_count_table(series, label_map, variable_name):
    valid_codes = list(label_map.keys())

    return pd.DataFrame({
        "variable": variable_name,
        "code": valid_codes + ["Missing"],
        "count": [int(series.eq(code).sum()) for code in valid_codes]
                 + [int((series.isna() | ~series.isin(valid_codes)).sum())],
        "meaning": list(label_map.values()) + ["Missing or invalid"]
    })

alcohol_marijuana_counts = pd.concat(
    [
        make_code_count_table(
            alcohol_raw,
            alcohol_label_map,
            "CurrentAlcoholUse"
        ),
        make_code_count_table(
            marijuana_raw,
            marijuana_label_map,
            "CurrentMarijuaUse"
        )
    ],
    ignore_index=True
)

alcohol_marijuana_counts.to_csv(
    TAB_DIR / "01_alcohol_marijuana_code_counts.csv",
    index=False
)

display(alcohol_marijuana_counts)

,variable,code,count,meaning
0,CurrentAlcoholUse,1,6946,0 days
1,CurrentAlcoholUse,2,2735,1 or 2 days
2,CurrentAlcoholUse,3,1369,3 to 5 days
3,CurrentAlcoholUse,4,839,6 to 9 days
4,CurrentAlcoholUse,5,555,10 to 19 days
5,CurrentAlcoholUse,6,105,20 to 29 days
6,CurrentAlcoholUse,7,120,all 30 days
7,CurrentAlcoholUse,Missing,1372,Missing or invalid
8,CurrentMarijuaUse,1,10868,0 times
9,CurrentMarijuaUse,2,1034,1 or 2 times


### 3.2 Health-risk Behavior Count

In [11]:
# ------------------------------------------------------------
# Health-risk behavior count
# Count is calculated only when all 3 indicators are valid
# ------------------------------------------------------------

health_risk_cols = [
    "smoker_binary",
    "alcohol_binary",
    "marijuana_binary"
]

all_three_valid = recoded[health_risk_cols].notna().all(axis=1)

sad_and_all_three_valid = recoded[
    ["sad_binary", "smoker_binary", "alcohol_binary", "marijuana_binary"]
].notna().all(axis=1)

recoded["health_risk_behavior_count"] = pd.NA

recoded.loc[all_three_valid, "health_risk_behavior_count"] = (
    recoded.loc[all_three_valid, health_risk_cols].sum(axis=1)
)

recoded["health_risk_behavior_count"] = recoded["health_risk_behavior_count"].astype("Int64")

# ------------------------------------------------------------
# Valid row check table
# ------------------------------------------------------------

health_risk_check = pd.DataFrame({
    "metric": [
        "3 indicators valid",
        "Sad + 3 indicators valid"
    ],
    "value": [
        int(all_three_valid.sum()),
        int(sad_and_all_three_valid.sum())
    ]
})

health_risk_check.to_csv(
    TAB_DIR / "01_health_risk_behavior_count_check.csv",
    index=False
)

display(health_risk_check)

# ------------------------------------------------------------
# Health-risk behavior count distribution
# ------------------------------------------------------------

health_risk_count_distribution = (
    recoded.loc[all_three_valid, "health_risk_behavior_count"]
    .value_counts()
    .reindex([0, 1, 2, 3], fill_value=0)
    .rename_axis("health_risk_behavior_count")
    .reset_index(name="count")
)

health_risk_count_distribution.to_csv(
    TAB_DIR / "01_health_risk_behavior_count_distribution.csv",
    index=False
)

display(health_risk_count_distribution)

,metric,value
0,3 indicators valid,12074
1,Sad + 3 indicators valid,12037


,health_risk_behavior_count,count
0,0,6212
1,1,3008
2,2,1615
3,3,1239


#### Interpretation

The three health-risk indicators are current cigarette use, current alcohol use, and current marijuana use.

- **3 indicators valid**: rows where all three health-risk indicators are valid.
- **Sad + 3 indicators valid**: rows where SadOrHopeless and all three health-risk indicators are valid.

The health-risk behavior count ranges from 0 to 3.

## 4. SAVE

In [12]:
q8_processed = save_processed_only(
    data=recoded,
    cols=[
        "sad_binary",
        "sad_group",
        "smoker_binary",
        "current_cigarette_status",
        "smoking_freq_group"
    ],
    required_cols=["sad_binary", "smoker_binary"],
    path=Q8_PROCESSED_PATH,
    label="Q8 processed-only data"
)

health_risk_processed = save_processed_only(
    data=recoded,
    cols=[
        "sad_binary",
        "sad_group",
        "smoker_binary",
        "alcohol_binary",
        "marijuana_binary",
        "health_risk_behavior_count"
    ],
    required_cols=[
        "sad_binary",
        "smoker_binary",
        "alcohol_binary",
        "marijuana_binary",
        "health_risk_behavior_count"
    ],
    path=HEALTH_RISK_PROCESSED_PATH,
    label="health-risk processed-only data"
)

Rows saved for Q8 processed-only data: 13174
Saved to: C:\Users\user\Downloads\2026-Spring-Stat2-Cycle3-main\data\processed\yrbs_cycle3_q8_processed_only.csv
Rows saved for health-risk processed-only data: 12037
Saved to: C:\Users\user\Downloads\2026-Spring-Stat2-Cycle3-main\data\processed\yrbs_cycle3_health_risk_processed_only.csv
